In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
def load_data() -> dict:
	"""Load all data

	:return: the data dictionary
	"""
	# load data
	dim_currency = pd.read_csv("../data/raw/dim_currency.csv")
	dim_market = pd.read_csv("../data/raw/dim_market.csv")
	dim_gateway = pd.read_csv("../data/raw/dim_gateway.csv")
	dim_merchant = pd.read_csv("../data/raw/dim_merchant.csv")
	dim_customer = pd.read_csv("../data/raw/dim_customer.csv")
	dim_date = pd.read_csv("../data/raw/dim_date.csv")
	fact_payment_intent = pd.read_csv("../data/raw/fact_payment_intent.csv")
	fact_payment_attempt = pd.read_csv("../data/raw/fact_payment_attempt.csv")
	fact_fee = pd.read_csv('../data/raw/fact_fee.csv')
	fact_refund = pd.read_csv("../data/raw/fact_refund.csv")
	fact_dispute = pd.read_csv("../data/raw/fact_dispute.csv")
	fact_settlement = pd.read_csv("../data/raw/fact_settlement.csv")
	fact_merchant_ledger = pd.read_csv("../data/raw/fact_merchant_ledger.csv")
	fact_fx_rate = pd.read_csv("../data/raw/fact_fx_rate.csv.gz")

	print("Data loaded")
	return {
		"dim_currency": dim_currency,
		"dim_market": dim_market,
		"dim_gateway": dim_gateway,
		"dim_merchant": dim_merchant,
		"dim_customer": dim_customer,
		"dim_date": dim_date,
		"fact_payment_intent": fact_payment_intent,
		"fact_payment_attempt": fact_payment_attempt,
		"fact_fee": fact_fee,
		"fact_refund": fact_refund,
		"fact_dispute": fact_dispute,
		"fact_settlement": fact_settlement,
		"fact_merchant_ledger": fact_merchant_ledger,
		"fact_fx_rate": fact_fx_rate
	}

data = load_data()

# load data
dim_currency = data["dim_currency"]
dim_market = data["dim_market"]
dim_gateway = data["dim_gateway"]
dim_merchant = data["dim_merchant"]
dim_customer = data["dim_customer"]
dim_date = data["dim_date"]
fact_payment_intent = data["fact_payment_intent"]
fact_payment_attempt = data["fact_payment_attempt"]
fact_fee = data["fact_fee"]
fact_refund = data["fact_refund"]
fact_dispute = data["fact_dispute"]
fact_settlement = data["fact_settlement"]
fact_merchant_ledger = data["fact_merchant_ledger"]
fact_fx_rate = data["fact_fx_rate"]

Data loaded


In [3]:
def preprocess_dim_merchant(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess the dim_merchant dataframe and save it as a csv to
    the processed folder.

	:param df: the dim_merchant dataframe
	:return: the info of the processed dataframe
	"""
    # copy the dataframe
    df = df.copy()

    # convert the columns to categorical
    df['country_code'] = df['country_code'].astype('category')
    df['industry'] = df['industry'].astype('category')
    df['settlement_currency'] = df['settlement_currency'].astype('category')
    df['pricing_tier'] = df['pricing_tier'].astype('category')
    df['payout_frequency'] = df['payout_frequency'].astype('category')
    df['account_status'] = df['account_status'].astype('category')

    # conver onboarded_at to datetime
    df['onboarded_at'] = pd.to_datetime(df['onboarded_at'])

    # save the processed dataframe as a csv
    df.to_csv('../data/processed/dim_merchant.csv', index=False)

    return df

preprocess_dim_merchant(dim_merchant).head()

,merchant_id,merchant_name,country_code,industry,onboarded_at,settlement_currency,pricing_tier,payout_frequency,account_status
0,1,FinPay Merchant 0001,GB,electronics,2023-03-11 00:00:00+00:00,GBP,starter,monthly,active
1,2,FinPay Merchant 0002,US,services,2019-11-20 00:00:00+00:00,USD,starter,monthly,active
2,3,FinPay Merchant 0003,US,retail,2025-11-29 00:00:00+00:00,USD,growth,weekly,active
3,4,FinPay Merchant 0004,US,electronics,2020-08-06 00:00:00+00:00,USD,starter,monthly,active
4,5,FinPay Merchant 0005,US,retail,2025-08-09 00:00:00+00:00,USD,starter,weekly,active


In [4]:
def preprocess_dim_customer(df):
    # copy the dataframe
    df = df.copy()

    # fix data types
    df['signed_up_at'] = pd.to_datetime(df['signed_up_at'])
    df['country_code'] = df['country_code'].astype('category')
    df['acquisition_channel'] = df['acquisition_channel'].astype('category')
    df['is_verified'] = df['is_verified'].astype('category')

    # save the dataframe
    df.to_csv('../data/processed/dim_customer.csv', index=False)
    return df

preprocess_dim_customer(dim_customer)

,customer_id,signed_up_at,country_code,acquisition_channel,is_verified
0,1,2020-08-27 01:00:00+00:00,US,referral,1
1,2,2020-08-27 01:00:00+00:00,FR,organic,1
2,3,2020-08-27 01:00:00+00:00,FR,paid_search,1
3,4,2020-08-27 01:00:00+00:00,RU,paid_search,1
4,5,2020-08-27 01:00:00+00:00,US,partner,1
...,...,...,...,...,...
99995,99996,2026-08-25 01:00:00+00:00,DE,paid_search,1
99996,99997,2026-08-25 01:00:00+00:00,GB,referral,1
99997,99998,2026-08-25 01:00:00+00:00,CN,social,1
99998,99999,2026-08-25 01:00:00+00:00,US,referral,1


In [5]:
def preprocess_dim_date(df):
    # copy the dataframe
    df = df.copy()

    # fix the date format
    df['date_utc'] = pd.to_datetime(df['date_utc'])
    df['year'] = df['date_utc'].dt.year
    df['day_of_year'] = df['date_utc'].dt.day_of_year
    df['month'] = df['date_utc'].dt.month
    df['month_name'] = df['date_utc'].dt.month_name()
    df['day_of_month'] = df['date_utc'].dt.day
    df['is_month_end'] = df['date_utc'].dt.is_month_end
    df['is_month_start'] = df['date_utc'].dt.is_month_start
    df['day_name'] = df['date_utc'].dt.day_name()
    df['day_of_week'] = df['date_utc'].dt.day_of_week
    df['quarter'] = df['date_utc'].dt.quarter
    df['is_quarter_end'] = df['date_utc'].dt.is_quarter_end
    df['is_quarter_start'] = df['date_utc'].dt.is_quarter_start
    # save the dataframe
    df.to_csv("../data/processed/dim_date.csv", index=False)

    return df

preprocess_dim_date(dim_date)

,date_key,date_utc,year,month,year_month,day_of_week,is_weekend,day_of_year,month_name,day_of_month,is_month_end,is_month_start,day_name,quarter,is_quarter_end,is_quarter_start
0,20211001,2021-10-01,2021,10,2021-10,4,0,274,October,1,False,True,Friday,4,False,True
1,20211002,2021-10-02,2021,10,2021-10,5,1,275,October,2,False,False,Saturday,4,False,False
2,20211003,2021-10-03,2021,10,2021-10,6,1,276,October,3,False,False,Sunday,4,False,False
3,20211004,2021-10-04,2021,10,2021-10,0,0,277,October,4,False,False,Monday,4,False,False
4,20211005,2021-10-05,2021,10,2021-10,1,0,278,October,5,False,False,Tuesday,4,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1821,20260926,2026-09-26,2026,9,2026-09,5,1,269,September,26,False,False,Saturday,3,False,False
1822,20260927,2026-09-27,2026,9,2026-09,6,1,270,September,27,False,False,Sunday,3,False,False
1823,20260928,2026-09-28,2026,9,2026-09,0,0,271,September,28,False,False,Monday,3,False,False
1824,20260929,2026-09-29,2026,9,2026-09,1,0,272,September,29,False,False,Tuesday,3,False,False


In [6]:
def preprocess_fact_payment_intent(df):
    # copy the dataframe
    df = df.copy()
    df = df.drop(columns="order_reference") # drop the order_reference column

    # fix the data types
    df.loc[df['customer_id'].isnull(), 'customer_id'] = 0 # 0 represents a customer who has not signed up
    df['customer_id'] = df['customer_id'].astype(int)
    df['created_at'] = pd.to_datetime(df['created_at'])
    df['buyer_country'] = df['buyer_country'].astype('category')
    df['checkout_channel'] = df['checkout_channel'].astype('category')
    df['presentment_currency'] = df['presentment_currency'].astype('category')
    df['final_outcome'] = df['final_outcome'].astype('category')
    df['closed_at'] = pd.to_datetime(df['closed_at'])
    df['captured_at'] = pd.to_datetime(df['captured_at'])
    df.loc[df['captured_settlement_minor'].isnull(), 'captured_settlement_minor'] = 0 # 0 represents a settlement that has not been captured
    df['captured_settlement_minor'] = df['captured_settlement_minor'].astype(int)

    # calculate the time differences
    # payment_resolution_time is the time difference between the closed_at and created_at
    df['payment_resolution_time'] = (df['closed_at'] - df['created_at']).dt.total_seconds()

    # time_to_capture is the time difference between the captured_at and created_at
    df['time_to_capture'] = (df['captured_at'] - df['created_at']).dt.total_seconds()

	# fix amount to be major
    currencies_to_fix = dim_currency.loc[dim_currency['minor_units'] == 2, 'currency_code'].values.tolist()
    df['amount_major'] = df.apply(lambda x: (x['amount_minor'] / 100) if x['presentment_currency'] in currencies_to_fix else x['amount_minor'], axis=1)
    df['captured_settlement_major'] = df.apply(lambda x: (x['captured_settlement_minor'] / 100) if x['presentment_currency'] in currencies_to_fix else x['captured_settlement_minor'], axis=1)

	# save the dataframe
    df.to_csv('../data/processed/fact_payment_intent.csv', index=False)

    return df

preprocess_fact_payment_intent(fact_payment_intent)

,intent_id,merchant_id,customer_id,created_at,buyer_country,checkout_channel,presentment_currency,amount_minor,final_outcome,closed_at,captured_at,captured_settlement_minor,payment_resolution_time,time_to_capture,amount_major,captured_settlement_major
0,1,590,416,2021-10-06 15:49:43+00:00,CN,mobile_web,CNY,21107,succeeded,2021-10-06 15:50:00+00:00,2021-10-06 15:49:58+00:00,21107,17.0,15.0,211.07,211.07
1,2,268,0,2021-10-23 08:01:17+00:00,FR,app,EUR,5188,succeeded,2021-10-23 08:01:34+00:00,2021-10-23 08:01:32+00:00,5188,17.0,15.0,51.88,51.88
2,3,148,0,2021-10-29 15:35:58+00:00,DE,web,EUR,1210,succeeded,2021-10-29 15:36:18+00:00,2021-10-29 15:36:16+00:00,1210,20.0,18.0,12.10,12.10
3,4,150,19367,2021-10-31 15:08:18+00:00,DE,web,EUR,1072,succeeded,2021-10-31 15:08:27+00:00,2021-10-31 15:08:25+00:00,1072,9.0,7.0,10.72,10.72
4,5,1127,4989,2021-10-05 08:46:50+00:00,US,web,USD,1938,succeeded,2021-10-05 08:47:07+00:00,2021-10-05 08:47:05+00:00,1938,17.0,15.0,19.38,19.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1619995,1619996,772,22790,2026-09-10 17:25:46+00:00,CN,app,CNY,17007,succeeded,2026-09-10 17:25:55+00:00,2026-09-10 17:25:53+00:00,17007,9.0,7.0,170.07,170.07
1619996,1619997,1381,0,2026-09-26 10:46:18+00:00,CA,web,CAD,6551,succeeded,2026-09-26 10:46:40+00:00,2026-09-26 10:46:38+00:00,6551,22.0,20.0,65.51,65.51
1619997,1619998,1588,21925,2026-09-24 02:08:40+00:00,GB,mobile_web,GBP,9509,succeeded,2026-09-24 02:09:30+00:00,2026-09-24 02:09:28+00:00,9509,50.0,48.0,95.09,95.09
1619998,1619999,1951,0,2026-09-30 05:59:05+00:00,GB,web,GBP,4022,succeeded,2026-09-30 05:59:29+00:00,2026-09-30 05:59:27+00:00,4022,24.0,22.0,40.22,40.22
